# Find Hard Cases for Annotation

This notebook identifies images where your model struggles, so Elizabeth can prioritize labeling the most valuable images.

**Strategy:**
1. Run inference on all available images (labeled + unlabeled)
2. Find images with low-confidence detections
3. Find images with no detections (potential missed vessels)
4. Find images where predictions differ significantly from ground truth
5. Export a prioritized list for annotation

## 1. Setup

In [1]:
from ultralytics import YOLO
import torch
from pathlib import Path
import json
import shutil
from datetime import datetime
import os

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA B200


In [2]:
# =============================================================================
# CONFIGURE PATHS
# =============================================================================

# Your best trained model
best_model_path = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_large_image/large_imgsz_25602/weights/best.pt"
# UPDATE THIS PATH to your actual best model!

# Directory with images to analyze (can include unlabeled images)
images_dir = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet"

# Your current training/val images (to compare predictions vs ground truth)
train_images_dir = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/images/train"
val_images_dir = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/images/val"
train_labels_dir = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/labels/train"
val_labels_dir = "/blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet/jupiter_inlet_yolo/labels/val"

# Output directory for hard cases
output_dir = Path("/blue/bsc4892/aileenlavelle/PBC_Object_Detection/hard_cases_for_annotation")
output_dir.mkdir(exist_ok=True)

# Verify paths
print("Checking paths...")
for path, name in [(best_model_path, "Best model"), (images_dir, "Images dir")]:
    if Path(path).exists():
        print(f"  ✅ {name}: {path}")
    else:
        print(f"  ❌ {name} NOT FOUND: {path}")

Checking paths...
  ✅ Best model: /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_large_image/large_imgsz_25602/weights/best.pt
  ✅ Images dir: /blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet


In [3]:
# Load model
print(f"Loading model: {best_model_path}")
model = YOLO(best_model_path)
print("✅ Model loaded")

Loading model: /blue/bsc4892/aileenlavelle/PBC_Object_Detection/runs/detect/jupiter_inlet_large_image/large_imgsz_25602/weights/best.pt
✅ Model loaded


## 2. Find All Images

In [4]:
# Find all images
image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif']

def find_images(directory):
    """Recursively find all images in a directory."""
    images = []
    for ext in image_extensions:
        images.extend(Path(directory).rglob(f'*{ext}'))
        images.extend(Path(directory).rglob(f'*{ext.upper()}'))
    return list(set(images))  # Remove duplicates

all_images = find_images(images_dir)
print(f"Found {len(all_images)} total images in {images_dir}")

# Show sample
print("\nSample images:")
for img in all_images[:5]:
    print(f"  {img.name}")

Found 2625 total images in /blue/bsc4892/aileenlavelle/PBC_Object_Detection/Jupiter_Inlet

Sample images:
  o041034x_jpg.rf.8aee8a56d8bb51d44e3310cb941c3ab6_slice_007.jpg
  o010951t_jpg.rf.ef3023d0847b76c15df66fc960158503_slice_001.jpg
  s301339s_jpg.rf.b25df926a885be06303732c12da6a241_slice_000.jpg
  s271159o_jpg.rf.e0a38fae6530d2022c15ae8203ca299e_slice_008.jpg
  s300946k_jpg.rf.fffef097f9b900391745aa8df102891c_slice_008.jpg


In [5]:
# Identify which images are already labeled (in train/val)
labeled_images = set()

for dir_path in [train_images_dir, val_images_dir]:
    if Path(dir_path).exists():
        for img in find_images(dir_path):
            labeled_images.add(img.stem)  # Just the filename without extension

print(f"Already labeled: {len(labeled_images)} images")

# Separate labeled vs unlabeled
unlabeled_images = [img for img in all_images if img.stem not in labeled_images]
print(f"Unlabeled images: {len(unlabeled_images)}")

Already labeled: 366 images
Unlabeled images: 1893


## 3. Run Inference on All Images

In [6]:
# Run inference and collect results
print("Running inference on all images...")
print("This may take a few minutes...\n")

inference_results = []

for i, img_path in enumerate(all_images):
    if (i + 1) % 50 == 0:
        print(f"  Processed {i + 1}/{len(all_images)} images...")
    
    try:
        # Run prediction
        results = model.predict(
            source=str(img_path),
            imgsz=2560,
            conf=0.1,  # Low threshold to catch uncertain detections
            verbose=False
        )
        
        result = results[0]
        boxes = result.boxes
        
        # Extract detection info
        num_detections = len(boxes)
        confidences = boxes.conf.cpu().numpy().tolist() if num_detections > 0 else []
        avg_confidence = sum(confidences) / len(confidences) if confidences else 0
        min_confidence = min(confidences) if confidences else 0
        max_confidence = max(confidences) if confidences else 0
        
        # Count low-confidence detections
        low_conf_count = sum(1 for c in confidences if c < 0.5)
        
        inference_results.append({
            'image_path': str(img_path),
            'image_name': img_path.name,
            'is_labeled': img_path.stem in labeled_images,
            'num_detections': num_detections,
            'confidences': confidences,
            'avg_confidence': avg_confidence,
            'min_confidence': min_confidence,
            'max_confidence': max_confidence,
            'low_conf_count': low_conf_count,
        })
        
    except Exception as e:
        print(f"  Error processing {img_path.name}: {e}")

print(f"\n✅ Processed {len(inference_results)} images")

Running inference on all images...
This may take a few minutes...

  Processed 50/2625 images...
  Processed 100/2625 images...
  Processed 150/2625 images...
  Processed 200/2625 images...
  Processed 250/2625 images...
  Processed 300/2625 images...
  Processed 350/2625 images...
  Processed 400/2625 images...
  Processed 450/2625 images...
  Processed 500/2625 images...
  Processed 550/2625 images...
  Processed 600/2625 images...
  Processed 650/2625 images...
  Processed 700/2625 images...
  Processed 750/2625 images...
  Processed 800/2625 images...
  Processed 850/2625 images...
  Processed 900/2625 images...
  Processed 950/2625 images...
  Processed 1000/2625 images...
  Processed 1050/2625 images...
  Processed 1100/2625 images...
  Processed 1150/2625 images...
  Processed 1200/2625 images...
  Processed 1250/2625 images...
  Processed 1300/2625 images...
  Processed 1350/2625 images...
  Processed 1400/2625 images...
  Processed 1450/2625 images...
  Processed 1500/2625 ima

## 4. Analyze Results & Find Hard Cases

In [7]:
# Categorize hard cases

# Category 1: No detections (potential missed vessels)
no_detections = [r for r in inference_results if r['num_detections'] == 0]

# Category 2: Low average confidence (model is uncertain)
low_avg_conf = [r for r in inference_results if r['avg_confidence'] > 0 and r['avg_confidence'] < 0.5]

# Category 3: Has low-confidence detections (borderline cases)
has_low_conf = [r for r in inference_results if r['low_conf_count'] > 0]

# Category 4: Very few detections (might be missing some)
few_detections = [r for r in inference_results if 0 < r['num_detections'] <= 2]

# Category 5: Unlabeled with detections (good candidates for pseudo-labeling)
unlabeled_with_detections = [r for r in inference_results if not r['is_labeled'] and r['num_detections'] > 0]

print("="*60)
print("HARD CASE ANALYSIS")
print("="*60)
print(f"\n📊 Total images analyzed: {len(inference_results)}")
print(f"   Already labeled: {sum(1 for r in inference_results if r['is_labeled'])}")
print(f"   Unlabeled: {sum(1 for r in inference_results if not r['is_labeled'])}")
print(f"\n🔍 Hard Cases Found:")
print(f"   No detections: {len(no_detections)} images")
print(f"   Low avg confidence (<0.5): {len(low_avg_conf)} images")
print(f"   Has uncertain detections: {len(has_low_conf)} images")
print(f"   Very few detections (1-2): {len(few_detections)} images")
print(f"\n📝 Unlabeled images with detections: {len(unlabeled_with_detections)}")

HARD CASE ANALYSIS

📊 Total images analyzed: 2625
   Already labeled: 732
   Unlabeled: 1893

🔍 Hard Cases Found:
   No detections: 486 images
   Low avg confidence (<0.5): 1253 images
   Has uncertain detections: 1598 images
   Very few detections (1-2): 1347 images

📝 Unlabeled images with detections: 1425


In [8]:
# Create priority score for each image
# Higher score = more valuable to annotate

for r in inference_results:
    score = 0
    reasons = []
    
    # Unlabeled images are more valuable (we don't have them yet)
    if not r['is_labeled']:
        score += 50
        reasons.append("unlabeled")
    
    # No detections - might have vessels we're missing
    if r['num_detections'] == 0:
        score += 30
        reasons.append("no_detections")
    
    # Low confidence detections - model is uncertain
    if r['low_conf_count'] > 0:
        score += 20 * r['low_conf_count']
        reasons.append(f"low_conf({r['low_conf_count']})")
    
    # Low average confidence
    if 0 < r['avg_confidence'] < 0.5:
        score += 25
        reasons.append("uncertain")
    
    # Few detections when we might expect more
    if 0 < r['num_detections'] <= 2:
        score += 10
        reasons.append("few_detections")
    
    r['priority_score'] = score
    r['priority_reasons'] = reasons

# Sort by priority
priority_sorted = sorted(inference_results, key=lambda x: x['priority_score'], reverse=True)

print("Top 20 Priority Images for Annotation:")
print("-"*80)
print(f"{'Rank':<5} {'Score':<7} {'Image':<40} {'Reasons':<30}")
print("-"*80)

for rank, r in enumerate(priority_sorted[:20], 1):
    reasons_str = ", ".join(r['priority_reasons'])
    print(f"{rank:<5} {r['priority_score']:<7} {r['image_name'][:40]:<40} {reasons_str:<30}")

Top 20 Priority Images for Annotation:
--------------------------------------------------------------------------------
Rank  Score   Image                                    Reasons                       
--------------------------------------------------------------------------------
1     265     a131312i_jpg.rf.f4b2bcabcd92b2223bc7c3ee low_conf(12), uncertain       
2     245     a101524q_jpg.rf.c9ecd8aa38b5494033de3bdf low_conf(11), uncertain       
3     245     o121232f_jpg.rf.7dafb7dcbbdf090325a7c240 low_conf(11), uncertain       
4     225     o101506n_jpg.rf.c5852c30d1fe9e68fc38a7b6 low_conf(10), uncertain       
5     205     a131412b_jpg.rf.97ec357feacb445dc044e63a low_conf(9), uncertain        
6     205     o111214p_jpg.rf.a59d3f71081176149f99115f low_conf(9), uncertain        
7     205     a081407j_jpg.rf.c2860542f576211f7f2fec4b low_conf(9), uncertain        
8     205     s071226f_jpg.rf.f20d7ee921295305ae7fe4d1 low_conf(9), uncertain        
9     205     l171523d_jp

## 5. Analyze Labeled Images (Find Where Model Fails)

In [9]:
def count_ground_truth_boxes(image_stem, labels_dirs):
    """Count number of ground truth boxes for an image."""
    for labels_dir in labels_dirs:
        label_path = Path(labels_dir) / f"{image_stem}.txt"
        if label_path.exists():
            with open(label_path, 'r') as f:
                lines = [l.strip() for l in f.readlines() if l.strip()]
                return len(lines)
    return None  # No label file found

# Compare predictions to ground truth for labeled images
labeled_results = [r for r in inference_results if r['is_labeled']]

comparison_results = []
for r in labeled_results:
    img_stem = Path(r['image_path']).stem
    gt_count = count_ground_truth_boxes(img_stem, [train_labels_dir, val_labels_dir])
    
    if gt_count is not None:
        pred_count = r['num_detections']
        diff = pred_count - gt_count
        
        comparison_results.append({
            'image_name': r['image_name'],
            'image_path': r['image_path'],
            'ground_truth': gt_count,
            'predictions': pred_count,
            'difference': diff,
            'avg_confidence': r['avg_confidence'],
            'min_confidence': r['min_confidence'],
        })

print(f"Compared predictions vs ground truth for {len(comparison_results)} labeled images")

Compared predictions vs ground truth for 732 labeled images


In [10]:
# Find images where model under-predicts (missing vessels)
under_predictions = [r for r in comparison_results if r['difference'] < 0]
under_predictions = sorted(under_predictions, key=lambda x: x['difference'])

# Find images where model over-predicts (false positives)
over_predictions = [r for r in comparison_results if r['difference'] > 0]
over_predictions = sorted(over_predictions, key=lambda x: x['difference'], reverse=True)

print("="*70)
print("PREDICTION vs GROUND TRUTH ANALYSIS")
print("="*70)

print(f"\n🔻 Model UNDER-predicts (missing vessels): {len(under_predictions)} images")
if under_predictions:
    print(f"{'Image':<40} {'GT':<5} {'Pred':<5} {'Missed':<7}")
    print("-"*60)
    for r in under_predictions[:10]:
        print(f"{r['image_name'][:40]:<40} {r['ground_truth']:<5} {r['predictions']:<5} {-r['difference']:<7}")

print(f"\n🔺 Model OVER-predicts (false positives): {len(over_predictions)} images")
if over_predictions:
    print(f"{'Image':<40} {'GT':<5} {'Pred':<5} {'Extra':<7}")
    print("-"*60)
    for r in over_predictions[:10]:
        print(f"{r['image_name'][:40]:<40} {r['ground_truth']:<5} {r['predictions']:<5} {r['difference']:<7}")

PREDICTION vs GROUND TRUTH ANALYSIS

🔻 Model UNDER-predicts (missing vessels): 79 images
Image                                    GT    Pred  Missed 
------------------------------------------------------------
s221855f_jpg.rf.fc6ec34c65c795128d7f015e 11    5     6      
s260718v_jpg.rf.cd439c56d61a88a4f0e07e0e 6     1     5      
s260703q_jpg.rf.c7af52d0117654328452a3ad 7     2     5      
s290920s_jpg.rf.466473ba26822d32cc902e12 6     1     5      
o021259v_jpg.rf.8d96ad45f47ca0fe8ca6630b 5     1     4      
a020821p_jpg.rf.ee92d968b82f7747734facb0 6     2     4      
o111525j_jpg.rf.adc85df0106dcea0eedda826 14    10    4      
s221053x_jpg.rf.1310a3143e40739806fac6c4 5     1     4      
s221758v_jpg.rf.637ec452822414fcdc5d53c7 6     2     4      
s300946k_jpg.rf.fffef097f9b900391745aa8d 6     3     3      

🔺 Model OVER-predicts (false positives): 379 images
Image                                    GT    Pred  Extra  
------------------------------------------------------------
a051

## 6. Export Hard Cases for Elizabeth

In [11]:
# Create output directories
priority_dir = output_dir / "1_high_priority"
medium_dir = output_dir / "2_medium_priority"
unlabeled_dir = output_dir / "3_unlabeled_with_detections"
under_pred_dir = output_dir / "4_model_missing_vessels"

for d in [priority_dir, medium_dir, unlabeled_dir, under_pred_dir]:
    d.mkdir(exist_ok=True)

print(f"Output directory: {output_dir}")

Output directory: /blue/bsc4892/aileenlavelle/PBC_Object_Detection/hard_cases_for_annotation


In [12]:
# Copy high priority images (top 50)
print("\nCopying high priority images...")
high_priority = [r for r in priority_sorted if r['priority_score'] >= 50][:50]

for r in high_priority:
    src = Path(r['image_path'])
    dst = priority_dir / src.name
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)

print(f"  Copied {len(high_priority)} high priority images to {priority_dir}")


Copying high priority images...
  Copied 50 high priority images to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/hard_cases_for_annotation/1_high_priority


In [13]:
# Copy medium priority images
print("\nCopying medium priority images...")
medium_priority = [r for r in priority_sorted if 20 <= r['priority_score'] < 50][:50]

for r in medium_priority:
    src = Path(r['image_path'])
    dst = medium_dir / src.name
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)

print(f"  Copied {len(medium_priority)} medium priority images to {medium_dir}")


Copying medium priority images...
  Copied 50 medium priority images to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/hard_cases_for_annotation/2_medium_priority


In [14]:
# Copy unlabeled images with high-confidence detections (good for pseudo-labeling)
print("\nCopying unlabeled images with detections...")
good_unlabeled = [r for r in unlabeled_with_detections if r['avg_confidence'] > 0.7][:30]

for r in good_unlabeled:
    src = Path(r['image_path'])
    dst = unlabeled_dir / src.name
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)

print(f"  Copied {len(good_unlabeled)} unlabeled images to {unlabeled_dir}")


Copying unlabeled images with detections...
  Copied 30 unlabeled images to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/hard_cases_for_annotation/3_unlabeled_with_detections


In [15]:
# Copy images where model misses vessels
print("\nCopying images where model misses vessels...")
for r in under_predictions[:20]:
    src = Path(r['image_path'])
    dst = under_pred_dir / src.name
    if src.exists() and not dst.exists():
        shutil.copy(src, dst)

print(f"  Copied {min(20, len(under_predictions))} images to {under_pred_dir}")


Copying images where model misses vessels...
  Copied 20 images to /blue/bsc4892/aileenlavelle/PBC_Object_Detection/hard_cases_for_annotation/4_model_missing_vessels


In [16]:
# Create summary report for Elizabeth
report = f"""# Hard Cases for Annotation - Summary Report
Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

## Overview
- Total images analyzed: {len(inference_results)}
- Already labeled: {sum(1 for r in inference_results if r['is_labeled'])}
- Unlabeled: {sum(1 for r in inference_results if not r['is_labeled'])}

## Priority Folders

### 1_high_priority ({len(high_priority)} images)
These are the MOST VALUABLE images to annotate. They include:
- Unlabeled images where model is uncertain
- Images with no detections (might have vessels we're missing)
- Images with low-confidence detections

### 2_medium_priority ({len(medium_priority)} images)
These are also valuable but slightly lower priority.

### 3_unlabeled_with_detections ({len(good_unlabeled)} images)
These are unlabeled images where the model found vessels with HIGH confidence.
The model's predictions are likely correct, so annotation should be quick.
Good for quickly expanding the dataset.

### 4_model_missing_vessels ({min(20, len(under_predictions))} images)
These are ALREADY LABELED images where the model predicts FEWER vessels 
than the ground truth. Worth reviewing to understand what the model misses.

## Recommendations
1. Start with 1_high_priority folder
2. Focus on variety: different lighting, weather, vessel sizes
3. For 3_unlabeled_with_detections: verify model predictions are correct, then approve
4. Review 4_model_missing_vessels to understand failure cases

## Current Model Performance
- mAP50: 0.8791
- Target: 0.90+ (need ~200+ more annotated images)
"""

report_path = output_dir / "README_FOR_ELIZABETH.md"
with open(report_path, 'w') as f:
    f.write(report)

print(f"\n📝 Summary report saved to: {report_path}")


📝 Summary report saved to: /blue/bsc4892/aileenlavelle/PBC_Object_Detection/hard_cases_for_annotation/README_FOR_ELIZABETH.md


In [ ]:
# Save detailed JSON for your records
detailed_results = {
    'generated': datetime.now().isoformat(),
    'model_path': best_model_path,
    'total_images': len(inference_results),
    'high_priority_images': [r['image_name'] for r in high_priority],
    'medium_priority_images': [r['image_name'] for r in medium_priority],
    'unlabeled_with_detections': [r['image_name'] for r in good_unlabeled],
    'under_predictions': [{'image': r['image_name'], 'gt': r['ground_truth'], 'pred': r['predictions']} 
                          for r in under_predictions[:20]],
    'statistics': {
        'no_detections': len(no_detections),
        'low_confidence': len(low_avg_conf),
        'under_predictions': len(under_predictions),
        'over_predictions': len(over_predictions),
    }
}

json_path = output_dir / "hard_cases_analysis.json"
with open(json_path, 'w') as f:
    json.dump(detailed_results, f, indent=2)

print(f"📊 Detailed analysis saved to: {json_path}")

## 7. Summary

In [ ]:
print("="*70)
print("ANNOTATION PRIORITY SUMMARY")
print("="*70)

print(f"\n📁 Output folder: {output_dir}")
print(f"\n📋 Folders created for Elizabeth:")
print(f"   1_high_priority/      - {len(high_priority)} images (START HERE)")
print(f"   2_medium_priority/    - {len(medium_priority)} images")
print(f"   3_unlabeled_with_detections/ - {len(good_unlabeled)} images (quick wins)")
print(f"   4_model_missing_vessels/     - {min(20, len(under_predictions))} images (review)")

print(f"\n📧 Message for Elizabeth:")
print("-"*50)
print(f"""Hi Elizabeth!

I've identified the most valuable images to annotate next.
They're in: {output_dir}

Please start with the '1_high_priority' folder - these are 
the images where our model is most uncertain.

The '3_unlabeled_with_detections' folder has images where 
the model already made predictions - you can verify and 
correct them, which should be faster than annotating from scratch.

Total: ~{len(high_priority) + len(medium_priority)} images to annotate.

Thanks!
Aileen""")

## 8. Optional: Visualize Hard Cases

In [ ]:
# Visualize some hard cases with predictions
VISUALIZE = True  # Set to True to generate visualizations

if VISUALIZE:
    viz_dir = output_dir / "visualizations"
    viz_dir.mkdir(exist_ok=True)
    
    print("Generating visualizations for top 10 hard cases...")
    
    for i, r in enumerate(high_priority[:10]):
        results = model.predict(
            source=r['image_path'],
            imgsz=2560,
            conf=0.1,
            save=True,
            project=str(viz_dir),
            name=f"hard_case_{i+1}",
            exist_ok=True
        )
    
    print(f"\n📸 Visualizations saved to: {viz_dir}")
    print("   Review these to understand why the model struggles with these images.")